In [2]:
import pandas as pd

In [3]:
train_df = pd.read_csv('./data/gsm8k_train.csv')
test_df = pd.read_csv('./data/gsm8k_test.csv')

In [47]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train_df[:10])
test_dataset = Dataset.from_pandas(test_df)

In [42]:
train_dataset[1]

{'question': 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?',
 'answer': 'Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.\nWorking 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.\n#### 10'}

In [14]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

import torch

In [10]:
model_dir="C:/Users/Gabriela/.cache/modelscope/hub/models/qwen/Qwen1___5-0___5B-Chat"

In [28]:
# ============================================================================
# 3. 4bit 量化（稳定不报错）
# ============================================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# ============================================================================
# 4. 加载模型 & Tokenizer
# ============================================================================
model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]c:\Users\Gabriela\anaconda3\envs\llm_env\lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\Gabriela\anaconda3\envs\llm_env\lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:05<00:00, 52.75it/s]


In [48]:
def format_example_batch(examples):
    # 直接批量取 question 和 answer，不用循环！
    questions = examples["question"]
    answers = examples["answer"]
    
    texts = [
        f"用户：{q}\n助手：{a}"
        for q, a in zip(questions, answers)
    ]
    return texts

# ============================================================================
# 6. Tokenize 【修复完成】
# ============================================================================
def tokenize_func(examples):
    texts = format_example_batch(examples)
    
    tok = tokenizer(
        texts,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt"
    )
    tok["labels"] = tok["input_ids"].clone()
    return tok

tokenized_train = train_dataset.map(
    tokenize_func,
    batched=True,
    remove_columns=train_dataset.column_names
)

Map: 100%|██████████| 10/10 [00:00<00:00, 1424.41 examples/s]


In [49]:
# ============================================================================
# 7. LoRA
# ============================================================================
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================================
# 8. 训练参数
# ============================================================================
training_args = TrainingArguments(
    output_dir="./qwen1.5-gsm8k-sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    optim="adamw_torch",
    report_to="none",
)

# ============================================================================
# 9. Trainer
# ============================================================================
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

trainable params: 1,572,864 || all params: 465,560,576 || trainable%: 0.3378


In [50]:
trainer.train()
model.save_pretrained("./qwen-gsm8k-lora-final")
print("训练完成！")

Step,Training Loss


训练完成！
